In [1]:
from pathlib import Path
import sys

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import CV_FOLDS, RANDOM_STATE
from src.data import (
    create_target,
    get_feature_sets,
    load_split_indices,
    load_student_data,
)
from src.preprocessing import build_preprocessor

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/keonidaniels/Intro to Ai final/student-performance-decision-support


In [2]:
df = create_target(load_student_data())

X_base, X_early, X_progress, y = get_feature_sets(df)

print("Dataset shape:", df.shape)
print("Target distribution:")
print(y.value_counts())

Dataset shape: (649, 34)
Target distribution:
needs_support
0    549
1    100
Name: count, dtype: int64


In [3]:
train_indices, test_indices = load_split_indices()

X_train_early = X_early.loc[train_indices]
X_test_early = X_early.loc[test_indices]

X_train_progress = X_progress.loc[train_indices]
X_test_progress = X_progress.loc[test_indices]

y_train = y.loc[train_indices]
y_test = y.loc[test_indices]

print("Train rows:", len(train_indices))
print("Test rows :", len(test_indices))

Train rows: 519
Test rows : 130


In [4]:
# Leakage and dependency checks

assert "G3" not in X_early.columns
assert "G3" not in X_progress.columns

assert "G1" not in X_early.columns
assert "G2" not in X_early.columns

assert "G1" in X_progress.columns
assert "G2" in X_progress.columns

assert len(train_indices) == 519
assert len(test_indices) == 130
assert set(train_indices).isdisjoint(set(test_indices))

assert list(X_train_early.index) == list(y_train.index)
assert list(X_train_progress.index) == list(y_train.index)

print("All leakage checks passed.")

All leakage checks passed.


In [5]:
numeric_features_early = (
    X_train_early.select_dtypes(include=["int64", "float64"])
    .columns
    .tolist()
)

categorical_features_early = (
    X_train_early.select_dtypes(include=["object"])
    .columns
    .tolist()
)

preprocessor_early = build_preprocessor(
    numeric_features_early,
    categorical_features_early,
)

print("Early numeric features     :", len(numeric_features_early))
print("Early categorical features :", len(categorical_features_early))

Early numeric features     : 13
Early categorical features : 17


/var/folders/mq/989v1jf904zgwckcs_qzptgh0000gn/T/ipykernel_44790/3991329773.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X_train_early.select_dtypes(include=["object"])


In [9]:
from src.preprocessing import build_preprocessor

preprocessor_early = build_preprocessor(
    numeric_features_early,
    categorical_features_early
)

print(preprocessor_early)

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'Medu', 'Fedu', 'traveltime',
                                  'studytime', 'failures', 'famrel', 'freetime',
                                  'goout', 'Dalc', 'Walc', 'health',
                                  'absences']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                    

## Early-Warning Logistic Regression (Unweighted Baseline)

In [6]:
baseline_early_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_early),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

baseline_early_pipeline.fit(X_train_early, y_train)

baseline_test_pred = baseline_early_pipeline.predict(X_test_early)

baseline_support_recall = recall_score(
    y_test,
    baseline_test_pred,
    pos_label=1,
)

print("Baseline support recall:", round(baseline_support_recall, 4))

Baseline support recall: 0.15


## Early-Warning Logistic Regression (Balanced)

In [7]:
logistic_early_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_early),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_early_pipeline.fit(X_train_early, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](30,)","['school','sex','age',...,'Walc','health','absences']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all r

In [8]:
train_pred_early = logistic_early_pipeline.predict(X_train_early)
test_pred_early = logistic_early_pipeline.predict(X_test_early)
test_prob_early = logistic_early_pipeline.predict_proba(X_test_early)[:, 1]

logistic_early_result = {
    "experiment_name": "early_warning",
    "model_name": "logistic_regression",
    "training_accuracy": accuracy_score(y_train, train_pred_early),
    "testing_accuracy": accuracy_score(y_test, test_pred_early),
    "support_precision": precision_score(
        y_test,
        test_pred_early,
        pos_label=1,
    ),
    "support_recall": recall_score(
        y_test,
        test_pred_early,
        pos_label=1,
    ),
    "support_f1": f1_score(
        y_test,
        test_pred_early,
        pos_label=1,
    ),
    "roc_auc": roc_auc_score(y_test, test_prob_early),
    "cv_mean": None,
    "cv_std": None,
    "confusion_matrix": confusion_matrix(
        y_test,
        test_pred_early,
    ),
}

logistic_early_result

{'experiment_name': 'early_warning',
 'model_name': 'logistic_regression',
 'training_accuracy': 0.8111753371868978,
 'testing_accuracy': 0.7692307692307693,
 'support_precision': 0.35294117647058826,
 'support_recall': 0.6,
 'support_f1': 0.4444444444444444,
 'roc_auc': 0.775909090909091,
 'cv_mean': None,
 'cv_std': None,
 'confusion_matrix': array([[88, 22],
        [ 8, 12]])}

## Early-Warning Cross-Validation (Training Data Only)

In [9]:
cv = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

cv_results_early = cross_validate(
    logistic_early_pipeline,
    X_train_early,
    y_train,
    cv=cv,
    scoring=["recall", "f1", "roc_auc"],
)

logistic_early_result["cv_mean"] = cv_results_early["test_recall"].mean()
logistic_early_result["cv_std"] = cv_results_early["test_recall"].std()

print("Early-warning CV support recall mean:", round(logistic_early_result["cv_mean"], 4))
print("Early-warning CV support recall std :", round(logistic_early_result["cv_std"], 4))
print("Early-warning CV F1 mean            :", round(cv_results_early["test_f1"].mean(), 4))
print("Early-warning CV ROC-AUC mean       :", round(cv_results_early["test_roc_auc"].mean(), 4))

Early-warning CV support recall mean: 0.625
Early-warning CV support recall std : 0.0559
Early-warning CV F1 mean            : 0.4395
Early-warning CV ROC-AUC mean       : 0.7592


## Progress-Informed Logistic Regression

In [10]:
numeric_features_progress = (
    X_train_progress.select_dtypes(include=["int64", "float64"])
    .columns
    .tolist()
)

categorical_features_progress = (
    X_train_progress.select_dtypes(include=["object"])
    .columns
    .tolist()
)

preprocessor_progress = build_preprocessor(
    numeric_features_progress,
    categorical_features_progress,
)

logistic_progress_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_progress),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_progress_pipeline.fit(X_train_progress, y_train)

/var/folders/mq/989v1jf904zgwckcs_qzptgh0000gn/T/ipykernel_44790/1826370211.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X_train_progress.select_dtypes(include=["object"])


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](32,)","['school','sex','age',...,'absences','G1','G2']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,32
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaini

In [11]:
train_pred_progress = logistic_progress_pipeline.predict(X_train_progress)
test_pred_progress = logistic_progress_pipeline.predict(X_test_progress)
test_prob_progress = logistic_progress_pipeline.predict_proba(X_test_progress)[:, 1]

logistic_progress_result = {
    "experiment_name": "progress_informed",
    "model_name": "logistic_regression",
    "training_accuracy": accuracy_score(y_train, train_pred_progress),
    "testing_accuracy": accuracy_score(y_test, test_pred_progress),
    "support_precision": precision_score(
        y_test,
        test_pred_progress,
        pos_label=1,
    ),
    "support_recall": recall_score(
        y_test,
        test_pred_progress,
        pos_label=1,
    ),
    "support_f1": f1_score(
        y_test,
        test_pred_progress,
        pos_label=1,
    ),
    "roc_auc": roc_auc_score(y_test, test_prob_progress),
    "cv_mean": None,
    "cv_std": None,
    "confusion_matrix": confusion_matrix(
        y_test,
        test_pred_progress,
    ),
}

logistic_progress_result

{'experiment_name': 'progress_informed',
 'model_name': 'logistic_regression',
 'training_accuracy': 0.9421965317919075,
 'testing_accuracy': 0.9076923076923077,
 'support_precision': 0.6538461538461539,
 'support_recall': 0.85,
 'support_f1': 0.7391304347826086,
 'roc_auc': 0.9372727272727273,
 'cv_mean': None,
 'cv_std': None,
 'confusion_matrix': array([[101,   9],
        [  3,  17]])}

## Progress-Informed Cross-Validation (Training Data Only)

In [12]:
cv_results_progress = cross_validate(
    logistic_progress_pipeline,
    X_train_progress,
    y_train,
    cv=cv,
    scoring=["recall", "f1", "roc_auc"],
)

logistic_progress_result["cv_mean"] = cv_results_progress["test_recall"].mean()
logistic_progress_result["cv_std"] = cv_results_progress["test_recall"].std()

print("Progress CV support recall mean:", round(logistic_progress_result["cv_mean"], 4))
print("Progress CV support recall std :", round(logistic_progress_result["cv_std"], 4))
print("Progress CV F1 mean            :", round(cv_results_progress["test_f1"].mean(), 4))
print("Progress CV ROC-AUC mean       :", round(cv_results_progress["test_roc_auc"].mean(), 4))

Progress CV support recall mean: 0.825
Progress CV support recall std : 0.0729
Progress CV F1 mean            : 0.7157
Progress CV ROC-AUC mean       : 0.9544


## Final Comparison Table

In [13]:
results_df = pd.DataFrame(
    [
        logistic_early_result,
        logistic_progress_result,
    ]
)

summary_columns = [
    "experiment_name",
    "model_name",
    "training_accuracy",
    "testing_accuracy",
    "support_precision",
    "support_recall",
    "support_f1",
    "roc_auc",
    "cv_mean",
    "cv_std",
]

results_df[summary_columns]

,experiment_name,model_name,training_accuracy,testing_accuracy,support_precision,support_recall,support_f1,roc_auc,cv_mean,cv_std
0,early_warning,logistic_regression,0.811175,0.769231,0.352941,0.60,0.444444,0.775909,0.625,0.055902
1,progress_informed,logistic_regression,0.942197,0.907692,0.653846,0.85,0.739130,0.937273,0.825,0.072887


In [14]:
print("Early-warning confusion matrix")
print(logistic_early_result["confusion_matrix"])

print("\nProgress-informed confusion matrix")
print(logistic_progress_result["confusion_matrix"])

logistic_progress_result

Early-warning confusion matrix
[[88 22]
 [ 8 12]]

Progress-informed confusion matrix
[[101   9]
 [  3  17]]


{'experiment_name': 'progress_informed',
 'model_name': 'logistic_regression',
 'training_accuracy': 0.9421965317919075,
 'testing_accuracy': 0.9076923076923077,
 'support_precision': 0.6538461538461539,
 'support_recall': 0.85,
 'support_f1': 0.7391304347826086,
 'roc_auc': 0.9372727272727273,
 'cv_mean': np.float64(0.825),
 'cv_std': np.float64(0.07288689868556625),
 'confusion_matrix': array([[101,   9],
        [  3,  17]])}

## Leakage Prevention

This corrected workflow prevents data leakage by:

- creating the target `needs_support` from `G3`,
- removing `G3` from both feature sets,
- excluding `G1` and `G2` from the early-warning experiment,
- loading the **saved shared train/test indices** rather than creating a new split,
- fitting imputers, encoders, and scalers **inside each training pipeline only**,
- performing **cross-validation on the training data only**, and
- reserving the held-out test set exclusively for final evaluation.

## Conclusion

Two logistic regression experiments were developed:

1. **Early-Warning Model** — uses only background and behavioral features and excludes `G1`, `G2`, and `G3`.
2. **Progress-Informed Model** — includes `G1` and `G2` while still excluding `G3`.

The balanced logistic regression approach was used to improve the detection of students who may require academic support. Model performance was evaluated using accuracy, precision, recall, F1-score, ROC-AUC, confusion matrices, and cross-validation recall scores.

These results provide a baseline that can be compared with the Decision Tree and Random Forest models developed by other team members.


## Handoff Note for Selorm

- **Shared split:** reused through `load_split_indices()`.
- **Target definition:** `needs_support = (G3 < 10).astype(int)`.
- **Early-warning features:** exclude `G1`, `G2`, and `G3`.
- **Progress-informed features:** include `G1` and `G2`, exclude `G3`.
- **Cross-validation:** five-fold stratified CV on the **training partition only**.
- **Result schema:** both experiments expose the required shared fields with non-NaN `cv_mean` and `cv_std`.

After saving this notebook, use **Restart Kernel → Run All** to regenerate the final metrics and ensure no stale outputs remain before committing.